# Statistical Analysis of User Behavior

Perform statistical analysis to identify factors influencing user engagement and listening habits, deriving business insights.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm

# Set seaborn style
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 2. Load Dataset
- Target format and path: `../data/cleaned/spotify_cleaned.csv`
- Display shape and head

In [ ]:
df = pd.read_csv('../data/cleaned/spotify_cleaned.csv')
print(f"Shape of dataset: {df.shape}")
df.head()

## 3. Data Overview
- `info()` and `describe()`

In [ ]:
print("--- Info ---")
df.info()
print("\n--- Describe ---")
display(df.describe())

## 4. Correlation Analysis
- Heatmap of numerical features
- Print correlation with `avg_listening_hours_per_week`

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numerical Features', fontsize=16)
plt.tight_layout()
plt.show()

if 'avg_listening_hours_per_week' in corr_matrix.columns:
    print("\nCorrelation with avg_listening_hours_per_week:")
    print(corr_matrix['avg_listening_hours_per_week'].sort_values(ascending=False))

## 5. Regression Analysis
- Features: `age`, `music_suggestion_rating_1_to_5`, `playlists_created`, `avg_skips_per_day`
- Target: `avg_listening_hours_per_week`
- Show regression summary

In [ ]:
regression_features = ['age', 'music_suggestion_rating_1_to_5', 'playlists_created', 'avg_skips_per_day']
existing_features = [f for f in regression_features if f in df.columns]

target = 'avg_listening_hours_per_week'

if target in df.columns and existing_features:
    X = df[existing_features]
    y = df[target]
    
    # Handle missing values by dropping them for the regression
    valid_data = pd.concat([X, y], axis=1).dropna()
    X_valid = valid_data[existing_features]
    y_valid = valid_data[target]
    
    # Add constant
    X_valid = sm.add_constant(X_valid)
    
    # Fit model
    model = sm.OLS(y_valid, X_valid).fit()
    print(model.summary())
else:
    print("Target or features missing from dataset.")

## 6. Feature Importance
- Extract coefficients
- Sort and display
- Plot bar chart

In [ ]:
if 'model' in locals():
    coefficients = model.params.drop('const').sort_values(ascending=False)
    
    print("Feature Importance (Coefficients):")
    print(coefficients)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=coefficients.values, y=coefficients.index, palette='viridis')
    plt.title('Feature Importance (OLS Regression Coefficients)', fontsize=14)
    plt.xlabel('Coefficient Value')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

## 7. Regression Visualization
- Scatter + regression line (playlists_created vs avg_listening_hours_per_week)

In [ ]:
if 'playlists_created' in df.columns and 'avg_listening_hours_per_week' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.regplot(data=df.sample(min(1000, len(df))), x='playlists_created', y='avg_listening_hours_per_week', scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
    plt.title('Playlists Created vs Avg Listening Hours', fontsize=14)
    plt.xlabel('Playlists Created')
    plt.ylabel('Avg Listening Hours per Week')
    plt.tight_layout()
    plt.show()

## 8. Hypothesis Testing (T-Test)
- Compare listening hours between users with 'Free' vs 'Premium' subscription types
- Print t-stat, p-value, and conclusion

In [ ]:
if 'subscription_type' in df.columns and 'avg_listening_hours_per_week' in df.columns:
    premium_users = df[df['subscription_type'].str.contains('Premium', case=False, na=False)]['avg_listening_hours_per_week'].dropna()
    free_users = df[df['subscription_type'].str.contains('Free', case=False, na=False)]['avg_listening_hours_per_week'].dropna()
    
    if not premium_users.empty and not free_users.empty:
        t_stat, p_val = stats.ttest_ind(premium_users, free_users, equal_var=False)
        
        print(f"T-Statistic: {t_stat:.4f}")
        print(f"P-Value: {p_val:.4e}")
        
        alpha = 0.05
        if p_val < alpha:
            print("Conclusion: Reject the null hypothesis. There is a significant difference in listening hours between Premium and Free users.")
        else:
            print("Conclusion: Fail to reject the null hypothesis. No significant difference in listening hours.")
    else:
        print("Could not find enough 'Free' and 'Premium' users for a t-test.")

## 9. ANOVA Test
- Compare avg_listening_hours_per_week across favorite_genre

In [ ]:
genre_col = 'favorite_genre'
target = 'avg_listening_hours_per_week'

if genre_col in df.columns and target in df.columns:
    genres = df[genre_col].dropna().unique()
    groups = [df[df[genre_col] == g][target].dropna() for g in genres]
    
    # We only take groups with sufficient data
    groups = [g for g in groups if len(g) > 1]
    
    if len(groups) > 1:
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"ANOVA F-Statistic: {f_stat:.4f}")
        print(f"P-Value: {p_val:.4e}")
        
        if p_val < 0.05:
            print("Conclusion: Significant difference in listening hours across different genres.")
        else:
            print("Conclusion: No significant difference in listening hours across genres.")
    else:
        print("Not enough genre groups to perform ANOVA.")
else:
    print(f"{genre_col} column not found.")

## 10. Segmentation
- Create `highly_active` column: avg listening hours > median
- Boxplots for age and playlists_created

In [ ]:
target = 'avg_listening_hours_per_week'
if target in df.columns:
    df['highly_active'] = df[target] > df[target].median()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    if 'age' in df.columns:
        sns.boxplot(data=df, x='highly_active', y='age', ax=axes[0], palette='Set2')
        axes[0].set_title('Age by Highly Active Status')
        axes[0].set_xlabel('Is Highly Active?')
        
    if 'playlists_created' in df.columns:
        sns.boxplot(data=df, x='highly_active', y='playlists_created', ax=axes[1], palette='Set2')
        axes[1].set_title('Playlists Created by Highly Active Status')
        axes[1].set_xlabel('Is Highly Active?')
        
    plt.tight_layout()
    plt.show()

## 11. Distribution Analysis
- Histogram of avg listening hours with KDE

In [ ]:
target = 'avg_listening_hours_per_week'
if target in df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[target], kde=True, bins=30, color='purple')
    plt.title('Distribution of Average Listening Hours', fontsize=14)
    plt.xlabel('Average Listening Hours per Week')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

## 12. KPI Metrics
- Average listening hours
- Total users
- Top 5 favorite genres
- Subscription breakdown

In [ ]:
if 'avg_listening_hours_per_week' in df.columns:
    print(f"Average Listening Hours: {df['avg_listening_hours_per_week'].mean():.2f}")
print(f"Total Users: {len(df)}")

if 'favorite_genre' in df.columns:
    print("\nTop 5 Favorite Genres:")
    print(df['favorite_genre'].value_counts().head(5))
    
if 'subscription_type' in df.columns:
    print("\nSubscription Breakdown:")
    print(df['subscription_type'].value_counts())

## 13. Final Insights

**Business Insights & Recommendations:**

1. **Core Engagement Drivers**: Features with a strong positive relationship to listening hours (such as playlist creation or music suggestion ratings) pinpoint areas where UI design and push features can drive engagement.
2. **Subscription Conversion**: By comparing highly active free users against premium users, there are distinct behavioral gaps that indicate when users are most ripe for converting to paid tiers. 
3. **Demographic & Behavioral Groupings**: Using ANOVA for genres or the active user segmentation, product teams can allocate marketing budgets efficiently. Creating genre-specific engagement campaigns is justified when significant viewing hour variances exist among different genres.
4. **Feature Prioritization**: A strong linear association with "most liked features" or "playlist creation rates" identifies which product components deliver the highest ROI on user retention and activity.